# Tiled polygonization tutorial (CDL workflow)

Notebook walkthrough for `examples/cdl_tiled_polygonize.py`.

This notebook runs a local synthetic raster end-to-end, then includes an
optional real USDA CDL run.

In [ ]:
from pathlib import Path
import importlib.util
import tempfile

import numpy as np
import rasterio
from rasterio.transform import from_origin

def load_module(path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module

cdl_mod = load_module(Path('examples/cdl_tiled_polygonize.py'), 'cdl_tiled_polygonize_script')
print('Loaded examples/cdl_tiled_polygonize.py')

## 1) Create a synthetic categorical raster

In [ ]:
rng = np.random.default_rng(42)
synthetic = rng.integers(1, 8, size=(256, 256), dtype=np.uint8)
transform = from_origin(500000.0, 4500000.0, 30.0, 30.0)

tmp_dir = Path(tempfile.mkdtemp(prefix='contourrs-cdl-'))
raster_path = tmp_dir / 'synthetic_cdl.tif'

with rasterio.open(
    raster_path,
    'w',
    driver='GTiff',
    height=synthetic.shape[0],
    width=synthetic.shape[1],
    count=1,
    dtype=synthetic.dtype,
    crs='EPSG:32615',
    transform=transform,
    nodata=0,
) as dst:
    dst.write(synthetic, 1)

print(f'Wrote synthetic raster: {raster_path}')

## 2) Polygonize in tiles, then merge class-matching neighbors

In [ ]:
tiled = cdl_mod.polygonize_tiled(raster_path, tile_size=64, connectivity=4)
merged = cdl_mod.merge_touching_same_class(tiled)

output_path = tmp_dir / 'synthetic_cdl_merged.parquet'
merged.to_parquet(output_path, index=False)

print(f'Raw tiled polygons: {len(tiled):,}')
print(f'Merged polygons: {len(merged):,}')
print(f"Classes in output: {merged['value'].nunique():,}")
print(f'Wrote merged parquet: {output_path}')

## 3) Render side-by-side raster vs polygons

In [ ]:
cdl_mod.plot_result(raster_path, merged)
print('Wrote assets/cdl_polygonize.png and docs/assets/cdl_polygonize.png')

## 4) Optional real USDA CDL run

Disabled by default to keep CI deterministic.
Set `RUN_REAL = True` to fetch and process real data.

In [ ]:
RUN_REAL = False

if RUN_REAL:
    year = 2023
    fips = '19153'
    download_dir = Path('examples/data')
    real_raster = download_dir / f'cdl_{year}_{fips}.tif'

    cdl_url = cdl_mod.resolve_cdl_url(year=year, fips=fips)
    cdl_mod.download_if_missing(cdl_url, real_raster)

    tiled_real = cdl_mod.polygonize_tiled(real_raster, tile_size=1024, connectivity=4)
    merged_real = cdl_mod.merge_touching_same_class(tiled_real)
    print(f'Real merged polygons: {len(merged_real):,}')
else:
    print('Skipping optional real-data run (set RUN_REAL = True to enable).')